# 02 · Training

This notebook trains the CNN-RNN captioning model on the full COCO 2014 training set (414k captions) for 3 epochs. The released weights `encoder-3.pkl` / `decoder-3.pkl` come from the run below. **The outputs are from that original run on a GPU.** Re-running the notebook overwrites `training_log.txt` and the files in `models/`.

## Design decisions

### Architecture

The model follows *Show and Tell* (Vinyals et al., 2015).

- **Encoder:** a ResNet-18 pretrained on ImageNet and kept frozen. Its classifier is removed, and a trainable linear layer maps the pooled 512-d features to a 512-d image embedding.
- **Decoder:** a single-layer LSTM with a 512-d hidden state. At the first time step it receives the image embedding. After that it receives 512-d word embeddings of the caption (`nn.Embedding`). A linear layer maps each hidden state to scores over the ~10k-token vocabulary.

`embed_size` and `hidden_size` are both 512, as in the paper. `vocab_threshold = 4` keeps the vocabulary at about 10k words while covering more than 99% of word occurrences (see `01_data_exploration`).

### Data augmentation

Images are resized so the shorter side is 256 and then randomly cropped to 224×224. This handles COCO's varying image sizes, gives ResNet the input size it expects, and adds some translation augmentation. I added `RandomHorizontalFlip()` on top of that. A mirrored image almost always fits the same caption, so the flip is a nearly free way to reduce overfitting. Normalization uses the ImageNet statistics, because those are what the frozen backbone was trained on.

### What is trained and what is frozen

All of the decoder (embedding, LSTM, output layer) is trained, and from the encoder only the new projection layer. The ResNet backbone stays frozen. This keeps its ImageNet features, makes each step much cheaper (no backward pass through the CNN), and lowers memory use. It also limits overfitting in a short 3-epoch run. The projection layer and the decoder start from random weights, so they have to be learned.

### Optimizer and batching

Adam with a learning rate of 1e-3. Per-parameter adaptive learning rates and momentum make LSTMs converge quickly and reliably with little tuning, and this standard default gave a steady drop in loss. To fit in memory, each forward pass uses 64 images, and gradients are accumulated over 4 steps, for an **effective batch size of 256**.

## Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # make project modules importable

import config

import math
import nltk
import numpy as np
import torch
import torch.nn as nn
import torch.utils.data as data
from torchvision import transforms

from data_loader import get_loader
from model import EncoderCNN, DecoderRNN

nltk.download("punkt_tab", quiet=True)
device = config.DEVICE

# Hyperparameters
batch_size = 64             # images per forward pass
accumulation_steps = 4      # effective batch size = 64 * 4 = 256
embed_size = config.EMBED_SIZE    # 512
hidden_size = config.HIDDEN_SIZE  # 512
num_epochs = 3
save_every = 1              # save weights after every epoch
print_every = 100

transform_train = transforms.Compose([
    transforms.Resize(256),                          # shorter side -> 256
    transforms.RandomCrop(224),                      # random 224x224 crop
    transforms.RandomHorizontalFlip(),               # cheap, caption-preserving augmentation
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406),      # ImageNet statistics
                         (0.229, 0.224, 0.225)),
])

data_loader = get_loader(transform=transform_train,
                         mode="train",
                         batch_size=batch_size,
                         vocab_threshold=config.VOCAB_THRESHOLD,
                         vocab_file=str(config.VOCAB_FILE),
                         vocab_from_file=True,
                         cocoapi_loc=str(config.COCO_ROOT))
vocab_size = len(data_loader.dataset.vocab)

encoder = EncoderCNN(embed_size).to(device)
decoder = DecoderRNN(embed_size, hidden_size, vocab_size).to(device)

criterion = nn.CrossEntropyLoss()
# Train the whole decoder plus the encoder's projection layer; ResNet stays frozen.
params = list(decoder.parameters()) + list(encoder.embed.parameters())
optimizer = torch.optim.Adam(params, lr=1e-3)

total_step = math.ceil(len(data_loader.dataset.caption_lengths) / batch_size)

Vocabulary successfully loaded from vocab.pkl file!
loading annotations into memory...
Done (t=0.43s)
creating index...
index created!
Obtaining caption lengths for the subset...


100%|██████████| 414113/414113 [00:23<00:00, 17388.59it/s]


## Training loop

At each step, a caption length is sampled and a same-length batch is drawn (see `01_data_exploration`). The model runs with teacher forcing and the cross-entropy loss is computed. Gradients are accumulated, and the optimizer steps once every 4 batches. Loss and perplexity are logged to `training_log.txt` at every step, and the weights are saved after each epoch.

In [ ]:
f = open(config.TRAINING_LOG, "w")
optimizer.zero_grad()

for epoch in range(1, num_epochs + 1):
    for i_step in range(1, total_step + 1):
        # Sample a caption length, then a batch of captions with that length.
        indices = data_loader.dataset.get_train_indices()
        data_loader.batch_sampler.sampler = data.sampler.SubsetRandomSampler(indices=indices)
        images, captions = next(iter(data_loader))
        images, captions = images.to(device), captions.to(device)

        # Teacher-forced forward pass.
        features = encoder(images)
        outputs = decoder(features, captions)
        loss = criterion(outputs.view(-1, vocab_size), captions.view(-1))

        # Gradient accumulation: step the optimizer every `accumulation_steps` batches.
        (loss / accumulation_steps).backward()
        if i_step % accumulation_steps == 0 or i_step == total_step:
            optimizer.step()
            optimizer.zero_grad()

        stats = "Epoch [%d/%d], Step [%d/%d], Loss: %.4f, Perplexity: %5.4f" % (
            epoch, num_epochs, i_step, total_step, loss.item(), np.exp(loss.item()))
        print("\r" + stats, end="")
        f.write(stats + "\n")
        f.flush()
        if i_step % print_every == 0:
            print("\r" + stats)

    if epoch % save_every == 0:
        torch.save(decoder.state_dict(), config.MODELS_DIR / f"decoder-{epoch}.pkl")
        torch.save(encoder.state_dict(), config.MODELS_DIR / f"encoder-{epoch}.pkl")

f.close()

Epoch [1/3], Step [100/6471], Loss: 4.4317, Perplexity: 84.07716
Epoch [1/3], Step [200/6471], Loss: 3.9175, Perplexity: 50.27371
Epoch [1/3], Step [300/6471], Loss: 3.7626, Perplexity: 43.05929
Epoch [1/3], Step [400/6471], Loss: 3.3499, Perplexity: 28.4990
Epoch [1/3], Step [500/6471], Loss: 3.5727, Perplexity: 35.61332
Epoch [1/3], Step [600/6471], Loss: 3.1353, Perplexity: 22.9947
Epoch [1/3], Step [700/6471], Loss: 3.5404, Perplexity: 34.48069
Epoch [1/3], Step [800/6471], Loss: 3.1868, Perplexity: 24.2113
Epoch [1/3], Step [900/6471], Loss: 3.8104, Perplexity: 45.16811
Epoch [1/3], Step [1000/6471], Loss: 3.2593, Perplexity: 26.0326
Epoch [1/3], Step [1100/6471], Loss: 2.9064, Perplexity: 18.2909
Epoch [1/3], Step [1200/6471], Loss: 3.1375, Perplexity: 23.0455
Epoch [1/3], Step [1300/6471], Loss: 2.8645, Perplexity: 17.5408
Epoch [1/3], Step [1400/6471], Loss: 2.8522, Perplexity: 17.3261
Epoch [1/3], Step [1500/6471], Loss: 2.5974, Perplexity: 13.4290
Epoch [1/3], Step [1600/6471

## Training curve

The loss drops quickly in the first few hundred steps, which is when the decoder learns the basic grammar of a caption. It then keeps falling slowly through epochs 2 and 3: the 200-step moving average is ≈2.35 at the end of epoch 1, ≈2.14 after epoch 2, and **≈2.02** at the end of training (perplexity ≈ 7.5). The curve below is produced from `training_log.txt` by `scripts/plot_loss.py`.

![Training loss](../assets/loss_curve.png)

The loss is noisy from step to step because each batch has a different caption length: short captions are easier to predict than long ones.